In [7]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from openai import OpenAI 
client = OpenAI()

MODEL = "gpt-5.4-nano"

response = client.responses.create(
    model=MODEL,
    input="Say hello in one sentence."
)   

print(response.output_text)

Hello!


In [10]:
#Hay tres papeles:

#"system" = las normas (el manual)
#"user" = tú
#"assistant" = el agente

SYSTEM_PROMPT = """
You are a friendly travel advisor who helps people plan personalised trips.

Information you must collect from the user before making a plan:
- Destination
- Duration of the trip (number of days)
- Budget and currency
- Priorities in budget allocation (e.g., accommodation, food, activities)
- Interests and preferences (e.g., adventure, relaxation, culture, food)
Optional, ask only if useful: budget, travel dates, who they travel with, pace,
constraints (diet, mobility, etc.).

How to talk:
- Ask at most 2 questions at a time, and wait for the user to answer before asking more.
- Use a friendly and conversational tone.  
- Avoid asking questions that are not relevant to the user trip planning.
- Avoid asking questions that the user has already answered, once you have the
  required information, stop asking and write the plan.
- If the user gives contradictory or unclear information (for example
  "3 days" and later "a week"), ask them to clarify before writing the plan.

When you have the required information, write the plan with these sections:
- Trip title
- Trip summary (2-3 sentences).
- Day-by-day itinerary (morning, afternoon, evening for each day).
- Practical travel tips.
- Estimated budget per person in euros, split into accommodation, food, transport and activities.
- Optional: a list of recommended restaurants, cafes, and bars.
- Alternative itinerary tips to avoid crowds or save money.

Keep the itinerary realistic: don't overpack days, and consider travel time
between places. Only help with travel planning; politely decline other topics.
"""

GREETING = "Hi! I'm your travel advisor. Where would you like to go, and for how many days?"

In [11]:
# History es una lista que dice quien habla (System) y que dice (content). La primera entrada es el sistema, la segunda es el saludo del agente.

history = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "assistant", "content": GREETING}
]

# 1. Aqui apuntamos lo que dice el usuario. En este caso, el usuario dice que quiere ir a Tokyo por 27 días.

history.append({"role": "user", "content": "I want to go to Tokyo for 27 days."})   

# 2. Aqui llamamos a la API de OpenAI para que nos diga que contesta y pasamos el historial completo.

response = client.responses.create(
    model=MODEL,
    input=history
)

#Sacamos solo el texto de la respuesta y lo guardamos en una caja llamada reply.

reply = response.output_text

# Apuntar también lo que dijo el agente: Si no en la siguiente llamada no sabrá lo que dijo él mismo.

history.append({"role": "assistant", "content": reply})

#comprobar

print(reply)
print(len(history), "messages in the list")



Great choice—Tokyo for 27 days sounds amazing.

Next, what’s your budget and currency (rough estimate is totally fine), and how would you like to prioritize it (e.g., more for accommodation vs. more for food/activities)?
4 messages in the list


In [ ]:
# Ahora vamos a crear una función que nos permita reutuilizar el código anterior.

history = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "assistant", "content": GREETING}
]

def chat(user_text):
    # 1. Aqui apuntamos lo que dice el usuario. En este caso, el usuario dice que quiere ir a Tokyo por 27 días.
    history.append({"role": "user", "content": user_text})   

    # 2. Aqui llamamos a la API de OpenAI para que nos diga que contesta y pasamos el historial completo.
    response = client.responses.create(
        model=MODEL,
        input=history
    )

    #Sacamos solo el texto de la respuesta y lo guardamos en una caja llamada reply.
    reply = response.output_text

    # Apuntar también lo que dijo el agente: Si no en la siguiente llamada no sabrá lo que dijo él mismo.
    history.append({"role": "assistant", "content": reply})

    return reply

#Chat es la celda 3 empaquetada para reutilizarla. Ahora podemos llamar a la función chat() con el texto que queramos y nos devolverá la respuesta del agente.

print(chat("Rome, 3 days"))

Great—Rome for 3 days! 😊  
Next: what’s your budget and currency, and how would you like it prioritized (e.g., more on accommodation vs. activities vs. food)?


In [17]:
history = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "assistant", "content": GREETING},
]
print("Advisor:", GREETING)

while True:
    user_text = input("You: ")

    if user_text.lower() in ["quit", "exit"]:
        print("Advisor: Bye!Have a great trip!")
        break

    reply = chat(user_text)
    print("\nAdvisor:", reply, "\n")

Advisor: Hi! I'm your travel advisor. Where would you like to go, and for how many days?

Advisor: Great choice—Tokyo for 27 days is an amazing amount of time. 😊  
1) What budget range should I plan around (and in which currency)?  
2) What are your top priorities for how you spend it (e.g., accommodation, food, activities—feel free to rank or give percentages)? 


Advisor: Perfect—good to know your stay and flight are already booked. 😊  
1) For the remaining spending, what priorities should I allocate your ~2500€ toward (food, activities, local transport, shopping/tours)?  
2) What are your main interests in Tokyo (e.g., food, culture/temples, anime/pop culture, nightlife, day trips, shopping, museums), and what pace do you want (relaxed vs packed)? 


Advisor: ## Trip title: Tokyo for 27 Days — Temples, Contemporary Art & Retro Gaming

### Trip summary
With your stay and flight already booked, this 27-day plan focuses your budget on the essentials (food + local transport) plus target

# Testing

## Changing itinerary, giving all inputs once and asking for out of context questions.

In [18]:
history = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "assistant", "content": GREETING},
]
print("Advisor:", GREETING)

while True:
    user_text = input("You: ")

    if user_text.lower() in ["quit", "exit"]:
        print("Advisor: Bye!Have a great trip!")
        break

    reply = chat(user_text)
    print("\nAdvisor:", reply, "\n")

Advisor: Hi! I'm your travel advisor. Where would you like to go, and for how many days?

Advisor: Nice—Japan for 5 days sounds great, especially with sake and videogames in mind. 🍶🎮  
What’s your budget and currency, and how would you like to split it across accommodation, food, and activities (roughly % is fine)? 


Advisor: Great—Kyoto for 4 days, with temples and food, and a mid budget. 🍱⛩️  

To tailor it well:  
1) What’s your budget range and currency (per person)?  
2) Are you more interested in **sake tastings/breweries** or **just sake-friendly bars/izakaya**? 


Advisor: Got it—no dog. 😊  

One quick check so I can plan well:
1) What’s your budget range and currency (per person)?  
2) Do you want to spend time **on a sake brewery/tasting tour**, or mostly **sake at restaurants and izakaya bars**? 


Advisor: I can’t physically take your dog out, but I can suggest a quick, practical option based on where you are.

To confirm, you mean: **leave your dog outside for a short pee